# PbP Level 0 processing
This is a GFAS PbP processing routine where the PbP data format is contained but the datasets are being modified and saved for further use.

It uses the following datasets: 
- raw PbP observational data: the cloud event 11.-12.08.2024 and dust event 16.08.2024
- raw PbP laboratory data: 16. and 17. July 2024
- scale factors in PbP_processing/detector_scaling_factors

Input needed:
    - path of the data

Sub-routines needed:
- PbP_processing/GFAS101_processing
- GFAS_reading/GFAS101_read_in
- PbP_processing/loss_Calc_functions_GFAS101

Processing applied
- attenuation
- scaling towards theoretical values
- sizing
- losses
- diameter binning

Output: corrected datasets of
- Laboratory measurements 
- cloud_fingerprint: cloud event r_pol and r_F/B dataset
- dust_fingerprint: dust event r_pol and r_F/B dataset

Things that need to be done when applying to different data:
- calibration routine
- check (with manufacturer) if attenuation is needed
- change dates

author: Lea Haberstock, Stockholm University, Department of Environmental Science, Atmospheric Unit \
developed toether with: Darrel Baumgardner and Paul Zieger

contact: lea.haberstock@aces.su.se \
used in publication: Haberstock et al. 2026 (submitted to AMT) 

In [21]:
import sys  
import numpy as np
import pandas as pd
import glob2
import glob, os
import os
from io import BytesIO
import datetime as dt
import calendar
from pandas.errors import EmptyDataError
from scipy.ndimage import shift
import scipy
import metpy
from metpy.calc import wind_components,wind_speed,wind_direction
from metpy.units import units
from matplotlib.pyplot import figure
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as colors
import shapely 
import matplotlib.ticker as ticker
import matplotlib 
import PyMieScatt  as pms
from scipy.stats import linregress
import dask
import dask.dataframe as dd
from pathlib import Path

project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
from PbP_processing.GFAS101_processing import *
from calibrations.MieCalculations import *
from GFAS_reading.GFAS101_read_in import *

In [22]:
# get scaling factors for sizing

path = '../PbP_processing/detector_scaling_factors'
scales = (pd.read_csv(path, index_col = 0)).squeeze()

In [23]:
# bins for binning the Sizer Peak and Diameter
bins = np.logspace(np.log10(1e-7), np.log10(5.357364e-06), 31)
bi = np.array([0,3,5,9,14,21,50])

### Field data

In [24]:
path = '/static/StationData/Campaigns/CIS_Sonnblick2024/GFAS'
starttime = '20240811'
endtime = '20240812'
timefmt = '%Y%m%d'

cloud1 = readPbP101(path, starttime, starttime, timefmt, filetype='GFAS_PbP_Data', compute=False)
cloud1 = cloud1[cloud1['datetime'] >= pd.to_datetime('2024-08-11 21:36:00')]
cloud2 = readPbP101(path, endtime, endtime, timefmt, filetype='GFAS_PbP_Data', compute=False)
cloud2 = cloud2[cloud2['datetime'] <= pd.to_datetime('2024-08-12 00:35:00')]

cloud = dd.concat([cloud1, cloud2], ignore_index=True)
cloud = cloud.map_partitions(attenuation)
cloud = cloud.map_partitions(qual_filter)

cloud['scaled Sizer Peak'] = cloud['Sizer Peak'] * scales['forward']
cloud['scaled S'] = cloud['S Peak'] * scales['S']
cloud['scaled P'] = cloud['P Peak'] * scales['P']
cloud['r_pol'] = (cloud['scaled S'] - cloud['scaled P']) / (cloud['scaled S'] + cloud['scaled P'])
cloud['F/B'] = cloud['scaled Sizer Peak']/(cloud['scaled S'] + cloud['scaled P'])

cloud['binned Sizer Peak'] = cloud['scaled Sizer Peak'].map_partitions(pd.cut, bins=bins, labels=bins[1:], include_lowest=True, meta=pd.Series(dtype="category"))

meta = sizing_model(cloud._meta.copy())
cloud = cloud.map_partitions(sizing_model, meta=meta)
cloud['binned Diameter'] = cloud['Diameter'].map_partitions(pd.cut, bins = bi, labels  = bi[1:], include_lowest = True)
cloud = cloud.compute()


Scheduled 436 CSV and 0 ZIP files.
Scheduled 282 CSV and 0 ZIP files.


In [25]:
path = '/static/StationData/Campaigns/CIS_Sonnblick2024/GFAS'
starttime = '20240816'
endtime = '20240816'
timefmt = '%Y%m%d'

dust = readPbP101(path, starttime, endtime, timefmt, filetype='GFAS_PbP_Data', compute=False)

Scheduled 444 CSV and 0 ZIP files.


In [26]:
dust = dust[(dust['datetime'] >= pd.to_datetime('2024-08-16 03:00'))& (dust['datetime'] <= pd.to_datetime('2024-08-16 12:00'))]

dust = dust.map_partitions(attenuation)
dust = dust.map_partitions(qual_filter)
dust['scaled Sizer Peak'] = dust['Sizer Peak'] * scales['forward']
dust['scaled S'] = dust['S Peak'] * scales['S']
dust['scaled P'] = dust['P Peak'] * scales['P']
dust['r_pol'] = (dust['scaled S'] - dust['scaled P']) / (dust['scaled S'] + dust['scaled P'])
dust['F/B'] = dust['scaled Sizer Peak']/(dust['scaled S'] + dust['scaled P'])

bins = np.logspace(np.log10(1e-7), np.log10(5.357364e-06), 31)
dust['binned Sizer Peak'] = dust['scaled Sizer Peak'].map_partitions(pd.cut, bins=bins, labels=bins[1:], include_lowest=True, meta=pd.Series(dtype="category"))

meta = sizing_model(dust._meta.copy())
dust = dust.map_partitions(sizing_model, meta=meta)
dust['binned Diameter'] = dust['Diameter'].map_partitions(pd.cut, bins = bi, labels  = bi[1:], include_lowest = True)
dust = dust.compute()

#### save a subsection of the data because big

In [ ]:
cloud.loc[:,['r_pol','F/B','binned Diameter','Diameter']].to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_field/fingerprint_cloud.csv', index=False)

In [ ]:
dust.loc[:,['r_pol','F/B','binned Diameter','Diameter']].to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_field/fingerprint_dust.csv', index=False)

### Laboratory data

In [29]:
path = '/static/StationData/Campaigns/GFASLabExperiments2024'
starttime = '20240716'
endtime = '20240717'
timefmt = '%Y%m%d'
User = 'GFAS_User_Data'

PbP = readPbP101(path,starttime,endtime,timefmt)
PbP['Computer Time (sec)'] -= pd.Timedelta('1 min 10 sec').total_seconds() # we had a time shift here 

Scheduled 65 CSV and 0 ZIP files.


In [30]:
def experiment_timeselect(data, starttime, endtime):
    V1 = (pd.to_datetime(starttime)-pd.to_datetime(dt.datetime(1904,1,1))).total_seconds()
    V2 = (pd.to_datetime(endtime)-pd.to_datetime(dt.datetime(1904,1,1))).total_seconds()
    # find closest time in pbp data to timestamp
    dV1 = data.iloc[(data['Computer Time (sec)']-V1).abs().argsort()[:1]].index[0]
    dV2 = data.iloc[(data['Computer Time (sec)']-V2).abs().argsort()[:1]].index[0]
    return data.iloc[dV1:dV2,:]

In [31]:
# day 1 
# volc = volcanic dust, cell = cellulose (different numbers describe different times), glassbeads/gb = glass beads 15 µm)

volc1 = experiment_timeselect(PbP, '2024-07-16 13:36:20', '2024-07-16 13:37:30')
volc2 = experiment_timeselect(PbP, '2024-07-16 08:51', '2024-07-16 08:56')
cell1 = experiment_timeselect(PbP, '2024-07-16 13:34:40', '2024-07-16 13:35:20')
cell2 = experiment_timeselect(PbP, '2024-07-16 13:42:40', '2024-07-16 13:43:30')
cell3= experiment_timeselect(PbP, '2024-07-16 09:02:30', '2024-07-16 09:03')

volcano = pd.concat([volc1, volc2])
cellulose = pd.concat([cell1, cell2, cell3])


In [32]:
# day 2 glass beads 15 µm
gb0 = experiment_timeselect(PbP, '2024-07-17 09:22:45', '2024-07-17 09:22:46')
gb2 = experiment_timeselect(PbP, '2024-07-17 09:26:50', '2024-07-17 09:26:56')
gb3 = experiment_timeselect(PbP, '2024-07-17 09:28:03', '2024-07-17 09:28:07')

glassbeads  = pd.concat([gb0,  gb2, gb3])
glassbeads.reset_index(drop=True, inplace=True)

In [106]:
# day 2 water nebulized at setting minimum (mini), middle (mid) and maximum (maxi) settings

mini = experiment_timeselect(PbP, '2024-07-17 09:09:40', '2024-07-17 09:11:20')
mid = experiment_timeselect(PbP, '2024-07-17 09:11:30', '2024-07-17 09:13')

# use only maxi in the end because the other size distributions were not large enough for the GFAS calibration
max1 = experiment_timeselect(PbP, '2024-07-17 08:57:10', '2024-07-17 08:57:40')
max2 = experiment_timeselect(PbP, '2024-07-17 08:58:35', '2024-07-17 09:01:30')
max3 = experiment_timeselect(PbP, '2024-07-17 09:13:25', '2024-07-17 09:15:49')

water = pd.concat([max1, max2, max3])

In [ ]:

df = [volcano, cellulose, glassbeads, water]

for D in df:
    attenuation(D)
    qual_filter(D)
    sizing_model(D)
    D['scaled Sizer Peak'] = D['Sizer Peak'] * scales['forward']
    D['scaled S'] = D['S Peak'] * scales['S']
    D['scaled P'] = D['P Peak'] * scales['P']
    D['r_pol'] = (D['scaled S'] - D['scaled P']) / (D['scaled S'] + D['scaled P'])
    D['F/B'] = D['scaled Sizer Peak']/(D['scaled S'] + D['scaled P'])
    D['binned Sizer Peak'] = pd.cut(D['scaled Sizer Peak'], bins=bins, labels=bins[1:], include_lowest=True)
    D['binned Diameter'] = pd.cut(D['Diameter'], bins = bi, labels  = bi[1:], include_lowest = True)




In [ ]:
volcano.to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_lab/volcano.csv', index=False)
cellulose.to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_lab/cellulose.csv', index=False)
glassbeads.to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_lab/gb15.csv', index=False)
water.to_csv(f'/share/Lea/data/Paper1/Level_1/GFAS_lab/water.csv', index=False)